In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
jsonl_file = Path("results.jsonl")
pdf_file = Path(f"all_confusion_matrices_{timestamp}.pdf")


# Load every valid JSONL record
runs = []
with jsonl_file.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue

        try:
            runs.append(json.loads(line))
        except json.JSONDecodeError as error:
            print(f"Skipping invalid JSON on line {line_number}: {error}")

if not runs:
    raise ValueError(f"No valid records found in {jsonl_file.resolve()}")

# Create one PDF page per confusion matrix
with PdfPages(pdf_file) as pdf:
    for run_number, run_data in enumerate(runs, start=1):
        model_name = run_data.get("model", "Unknown model")
        activity_labels = run_data["classes"]
        cm_normalized = np.asarray(
            run_data["confusion_matrix_percent"],
            dtype=float,
        )

        fig, ax = plt.subplots(figsize=(11, 8.5))

        sns.heatmap(
            cm_normalized,
            annot=True,
            fmt=".1%",
            cmap="Blues",
            vmin=0.0,
            vmax=1.0,
            xticklabels=activity_labels,
            yticklabels=activity_labels,
            cbar_kws={"label": "Percentage"},
            ax=ax,
        )

        accuracy = run_data.get("overall_accuracy")
        macro_f1 = run_data.get("macro_f1")
        run_timestamp = run_data.get("timestamp", "")

        accuracy_text = (
            f"{accuracy:.4f}" if accuracy is not None else "N/A"
        )
        macro_f1_text = (
            f"{macro_f1:.4f}" if macro_f1 is not None else "N/A"
        )

        ax.set_title(
            f"Normalized Confusion Matrix — {model_name}\n"
            f"Overall Accuracy: {accuracy_text} | Macro F1: {macro_f1_text}\n"
            f"Run {run_number} — {run_timestamp}",
            fontsize=13,
            pad=12,
        )
        ax.set_xlabel("Predicted Activity")
        ax.set_ylabel("Actual Activity")

        plt.xticks(rotation=45, ha="right")
        plt.yticks(rotation=0)
        plt.tight_layout()

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Created PDF with {len(runs)} confusion matrices:")
print(pdf_file.resolve())

Created PDF with 19 confusion matrices:
C:\Users\Adrian\VSCode_Projects\machineLearning\HAML-ET\modelling\all_confusion_matrices_20261007_110726.pdf
